# Model Compression — Pruning & Quantisation

Make models smaller and faster for mobile/edge deployment.

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## Why Compress?

| Challenge | Without | With |
|---|---|---|
| Model size | ResNet50=97MB | MobileNetV2=14MB |
| Inference time | 100ms CPU | 10ms CPU |
| Mobile RAM | Won't fit | Fits |

**Techniques:** Pruning, Quantisation, Knowledge Distillation, Architecture Search

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
data = load_breast_cancer()
X_tr,X_te,y_tr,y_te = train_test_split(data.data,data.target,test_size=0.2,random_state=42)
ns = [100,50,25,10,5,2,1]
accs = []
for n in ns:
    rf = RandomForestClassifier(n_estimators=n,random_state=42)
    rf.fit(X_tr,y_tr); accs.append(accuracy_score(y_te,rf.predict(X_te)))
plt.plot(ns,accs,'o-',color='#6b21a8',lw=2)
plt.xlabel('Trees (model size proxy)'); plt.ylabel('Accuracy')
plt.title('Pruning: Model Size vs Accuracy Trade-off')
plt.grid(alpha=0.3); plt.show()
for n,a in zip(ns,accs): print(f"  {n:<4} trees: {a:.3f}")

## Quantisation

Reduce weight precision: float32 → int8

```
float32: 32 bits per weight  (0.347821...)
int8:    8 bits per weight   (89, mapped to -128..127)

4x smaller model  +  2-4x faster inference  +  ~1% accuracy loss

# PyTorch dynamic quantisation:
model_int8 = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)
```

## Knowledge Distillation

Train a **small student** to mimic a **large teacher**:

```
Teacher (ResNet50, 25M params)  →  Student (MobileNet, 3.4M params)
         Soft probability outputs       Learn from teacher, not hard labels
```

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
data = load_breast_cancer()
X_tr,X_te,y_tr,y_te = train_test_split(data.data,data.target,test_size=0.2,random_state=42)

# Teacher: large model
teacher = MLPClassifier(hidden_layer_sizes=(256,128,64),max_iter=500,random_state=42)
teacher.fit(X_tr,y_tr)
teacher_acc = accuracy_score(y_te,teacher.predict(X_te))

# Student without distillation
student = MLPClassifier(hidden_layer_sizes=(32,),max_iter=500,random_state=42)
student.fit(X_tr,y_tr)
student_acc = accuracy_score(y_te,student.predict(X_te))

# Student with distillation: train on teacher's soft probabilities
soft_targets = teacher.predict_proba(X_tr)[:,1]
distilled = MLPClassifier(hidden_layer_sizes=(32,),max_iter=1000,random_state=42)
distilled.fit(X_tr, soft_targets)
distilled_acc = accuracy_score(y_te,(distilled.predict(X_te)>0.5).astype(int))

print(f"Teacher (large):       {teacher_acc:.3f}")
print(f"Student (no distil):   {student_acc:.3f}")
print(f"Student (distilled):   {distilled_acc:.3f}  <- recovered {distilled_acc-student_acc:.3f}")

## Compression Summary

| Technique | Speedup | Accuracy loss |
|---|---|---|
| Pruning | 1.5-3x | <1% |
| Quantisation (int8) | 2-4x | ~1% |
| Knowledge Distillation | 5-10x | ~2-3% |
| MobileNet architecture | 5-10x | ~2-5% |
| Neural Architecture Search | varies | varies |

---
**Your turn:** Change student to `(64,32)`. Does distillation still outperform no-distillation?